# Interactive Parquet with HoloViews

This example creates a one-million-row Parquet file and connects HoloViews zoom and pan events to `datashader-browser`. Each viewport is aggregated locally in the browser worker; no Python server is involved.

In [ ]:
from pathlib import Path

import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq

from datashader_browser import ParquetSource
from datashader_browser.holoviews import datashade

In [ ]:
DATA_PATH = Path('interactive-points.parquet')
ROWS_PER_BATCH = 100_000
BATCHES = 10

if not DATA_PATH.exists():
    rng = np.random.default_rng(42)
    schema = pa.schema([('x', pa.float64()), ('y', pa.float64())])
    with pq.ParquetWriter(DATA_PATH, schema) as writer:
        for index in range(BATCHES):
            center = -1.4 if index % 2 == 0 else 1.4
            writer.write_table(
                pa.table(
                    {
                        'x': rng.normal(center, 0.65, ROWS_PER_BATCH),
                        'y': rng.normal(-center / 2, 0.8, ROWS_PER_BATCH),
                    },
                    schema=schema,
                )
            )

print(f'{BATCHES * ROWS_PER_BATCH:,} rows; {DATA_PATH.stat().st_size / 1_000_000:.1f} MB')

In [ ]:
latest = None

def report(progress):
    global latest
    latest = progress

source = ParquetSource(DATA_PATH, batch_size=100_000)
plot = datashade(
    source,
    'x',
    'y',
    x_range=(-4, 4),
    y_range=(-4, 4),
    width=900,
    height=550,
    on_progress=report,
)
plot

Use the wheel-zoom and pan tools above. HoloViews sends each new viewport to browser Python, which starts a fresh bounded Parquet scan and replaces the image. The current backend still scans the file sequentially; spatial pruning is a later optimization.